# 10 - Multi-PDF Token Splitting

## Purpose

This notebook splits all loaded PDF page documents into token-sized chunks.

The goal is to prepare multiple PDFs for embedding and vector indexing while preserving source metadata such as file name, page number, and document type.

## Input

This notebook reloads all PDFs from:

```text
/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader

pdf_folder_path = "/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs"

files_info = dbutils.fs.ls(pdf_folder_path)

pdf_files = [
    file_info.path.replace("dbfs:", "")
    for file_info in files_info
    if file_info.path.lower().endswith(".pdf")
]

print("Number of PDF files found:", len(pdf_files))

for pdf_file in pdf_files:
    print(pdf_file)

all_pdf_docs = []

for pdf_file in pdf_files:
    print("\nLoading PDF:")
    print(pdf_file)

    loader = PyPDFLoader(pdf_file)
    docs = loader.load()

    file_name = os.path.basename(pdf_file)

    for doc in docs:
        doc.metadata["source_file"] = file_name
        doc.metadata["source_path"] = pdf_file
        doc.metadata["page_number"] = doc.metadata.get("page")
        doc.metadata["document_type"] = "pdf"

    all_pdf_docs.extend(docs)

print("\nAll PDFs loaded successfully")
print("Total page documents loaded:", len(all_pdf_docs))

In [0]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

multi_pdf_token_docs = token_splitter.split_documents(all_pdf_docs)

print("Token chunks created:", len(multi_pdf_token_docs))

In [0]:
for i, doc in enumerate(multi_pdf_token_docs):
    doc.metadata["chunk_id"] = i

print("Chunk IDs added")
print("Total chunks:", len(multi_pdf_token_docs))

In [0]:
for i, doc in enumerate(multi_pdf_token_docs[:5]):
    print(f"\nChunk {i}")
    print("Metadata:")
    print(doc.metadata)
    print()
    print("Content preview:")
    print(doc.page_content[:800])
    print("-" * 80)

In [0]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

for i, doc in enumerate(multi_pdf_token_docs[:10]):
    token_count = len(encoding.encode(doc.page_content))
    print(f"Chunk {i} token count:", token_count)

In [0]:
from collections import Counter

chunk_counts = Counter(
    doc.metadata.get("source_file")
    for doc in multi_pdf_token_docs
)

print("Chunk summary by PDF:")
print("-" * 80)

for file_name, chunk_count in chunk_counts.items():
    print(f"{file_name}: {chunk_count} chunks")

In [0]:
if len(multi_pdf_token_docs) == 0:
    raise ValueError("No token chunks were created.")

required_metadata = ["source_file", "page_number", "document_type", "chunk_id"]

for field in required_metadata:
    missing_count = sum(
        1 for doc in multi_pdf_token_docs
        if field not in doc.metadata
    )

    print(f"Missing {field}: {missing_count}")

print("Validation complete")